# Online metalog vs. empirical CVaR in risk-averse bandits

**Question:** in a bandit that picks the arm with the best CVaR (average of the worst 5% of rewards),
does estimating each arm's CVaR with an `OnlineMetalog` beat the standard approach of computing it from all stored samples?

**Fair comparison:** every algorithm uses the *same* UCB exploration rule,
`index = CVaR estimate + c·sqrt(log t / n)`. Only the CVaR estimator changes:

| Algorithm | CVaR estimator | Memory per arm |
|---|---|---|
| CVaR-UCB (empirical) | mean of the worst 5% of all samples so far | grows with every pull |
| CVaR-UCB (sliding window) | same, over the last `W` samples | `W` samples |
| CVaR-UCB (online metalog) | `OnlineMetalog.cvar(0.05)` | constant (5 coefficients) |
| mean-UCB | ignores risk, uses the mean | grows |

Regret is measured in CVaR: each round costs `best arm's true CVaR − chosen arm's true CVaR`.

In [ ]:
!pip install -q --force-reinstall --no-deps git+https://github.com/NixonZ/Metalog-Distribution

: 

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from metalog import OnlineMetalog

ALPHA = 0.05

def empirical_cvar(x, alpha=ALPHA):
    x = np.sort(np.asarray(x))
    return x[: max(1, int(np.ceil(alpha * len(x))))].mean()

def true_cvar(gen, n=2_000_000, seed=12345):
    return empirical_cvar(gen(np.random.default_rng(seed), n))

## Estimators
Each arm object learns from the rewards of its own pulls and reports a CVaR estimate.

In [ ]:
class EmpiricalArm:                      # stores everything
    def __init__(self, seed=None): self.x = []
    def update(self, v): self.x.append(v)
    def cvar(self): return empirical_cvar(self.x)
    def mean(self): return float(np.mean(self.x))

class WindowArm(EmpiricalArm):           # stores the last W rewards
    W = 300
    def update(self, v):
        self.x.append(v)
        if len(self.x) > self.W: self.x.pop(0)

class MetalogArm:                        # constant memory, converges on a fixed distribution
    kwargs = dict(n_terms=5)
    def __init__(self, seed=None): self.m = OnlineMetalog(rng=seed, **self.kwargs)
    def update(self, v): self.m.update(v)
    def cvar(self): return self.m.cvar(ALPHA, grid_size=400)

class TrackingMetalogArm(MetalogArm):    # constant memory, keeps adapting to change
    kwargs = dict(n_terms=5, average=False, decay=0, lr=0.1)

def run_bandit(arms_at, T, make_arm, seed, c=1.0, risk=True, n_init=10):
    """arms_at(t) -> list of reward generators at round t. Returns cumulative CVaR regret and pull counts."""
    rng = np.random.default_rng(seed)
    K = len(arms_at(0)[0])
    arms = [make_arm(seed * 100 + k) for k in range(K)]
    est, pulls, regret = np.zeros(K), np.zeros(K, int), np.zeros(T)
    t = 0

    def pull(k):
        nonlocal t
        gens, cvars = arms_at(t)
        arms[k].update(float(gens[k](rng, 1)[0]))
        pulls[k] += 1
        est[k] = arms[k].cvar() if risk else arms[k].mean()
        regret[t] = cvars.max() - cvars[k]
        t += 1

    for k in range(K):
        for _ in range(n_init): pull(k)
    while t < T:
        pull(int(np.argmax(est + c * np.sqrt(np.log(t + 1) / pulls))))
    return np.cumsum(regret), pulls

def compare(arms_at, T, algorithms, seeds=20):
    results = {}
    for label, (make_arm, risk) in algorithms.items():
        start = time.time()
        runs = [run_bandit(arms_at, T, make_arm, s, risk=risk) for s in range(seeds)]
        R = np.array([r for r, _ in runs]); P = np.mean([p for _, p in runs], 0)
        results[label] = R
        print(f"{label:30s} final regret {R[:, -1].mean():7.1f} ± {R[:, -1].std() / np.sqrt(seeds):5.1f}"
              f"   avg pulls per arm {np.round(P).astype(int)}   ({time.time() - start:.0f}s)")
    return results

def plot_regret(results, title, change_at=None):
    plt.figure(figsize=(9, 4.5))
    for label, R in results.items():
        m, se = R.mean(0), R.std(0) / np.sqrt(len(R))
        plt.plot(m, label=label); plt.fill_between(np.arange(len(m)), m - se, m + se, alpha=0.2)
    if change_at: plt.axvline(change_at, color="k", ls=":", label="distribution change")
    plt.xlabel("round"); plt.ylabel("cumulative CVaR regret"); plt.title(title); plt.legend(); plt.show()

## Experiment 1: fixed arms
The arm with the highest **mean** (risky, heavy-tailed) is not the arm with the best **CVaR** (safe).

In [ ]:
ARMS = [
    ("safe     N(0.5, 0.3)",       lambda r, n: r.normal(0.5, 0.3, n)),
    ("risky    1 + 0.6·t(3)",      lambda r, n: 1.0 + 0.6 * r.standard_t(3, n)),
    ("skewed   0.8 − Gumbel(0.5)", lambda r, n: 0.8 - r.gumbel(0, 0.5, n)),
    ("steady   N(0.2, 0.2)",       lambda r, n: r.normal(0.2, 0.2, n)),
]
gens = [g for _, g in ARMS]
cvars = np.array([true_cvar(g) for g in gens])
for (name, g), cv in zip(ARMS, cvars):
    print(f"{name:28s} mean {g(np.random.default_rng(0), 10**6).mean():6.3f}   CVaR5% {cv:6.3f}")

stationary = lambda t: (gens, cvars)
res1 = compare(stationary, T=5000, algorithms={
    "CVaR-UCB (empirical)":      (EmpiricalArm, True),
    "CVaR-UCB (online metalog)": (MetalogArm, True),
    "mean-UCB (risk-neutral)":   (EmpiricalArm, False),
})
plot_regret({k: v for k, v in res1.items() if not k.startswith("mean")}, "Fixed arms (mean-UCB omitted: its regret is ~50× larger)")

## Experiment 2: the best arm turns dangerous
Halfway through, the safe arm starts crashing: 8% of the time it loses 3.
Its CVaR drops far below the steady arm's, so the best decision changes.

Both forgetting settings were swept before fixing them here: the window size (W = 100, 300, 1000 gave regret 205–222)
and the tracking metalog's learning rate (0.02 → 690, 0.05 → 385, 0.1 → 260, 0.2 → 256). The best of each is used below.

In [ ]:
T2, CHANGE = 6000, 3000
crash = lambda r, n: r.normal(0.5, 0.3, n) - 3.0 * (r.random(n) < 0.08)
gens_after = [crash] + gens[1:]
cvars_after = np.array([true_cvar(crash)] + list(cvars[1:]))
print(f"safe arm CVaR: before {cvars[0]:.3f}, after {cvars_after[0]:.3f}  (steady arm: {cvars[3]:.3f})")

changing = lambda t: (gens, cvars) if t < CHANGE else (gens_after, cvars_after)
res2 = compare(changing, T=T2, algorithms={
    "CVaR-UCB (empirical)":           (EmpiricalArm, True),
    f"CVaR-UCB (window W={WindowArm.W})": (WindowArm, True),
    "CVaR-UCB (online metalog)":      (MetalogArm, True),
    "CVaR-UCB (tracking metalog)":    (TrackingMetalogArm, True),
})
plot_regret(res2, "Best arm turns dangerous at t = 3000", change_at=CHANGE)

## What this shows

**Fixed arms:** the online metalog matches empirical CVaR-UCB, within noise of each other,
while storing 5 coefficients per arm instead of every reward. Both are vastly better than
mean-UCB, which locks onto the high-mean, heavy-tailed arm.

**Changing arms:** a metalog that keeps averaging never forgets, and is the worst option here.
The tracking metalog recovers, but a sliding window over the last `W` rewards is still somewhat better.
Part of the gap is the model: "normal, with an 8% chance of losing 3" has two clusters of rewards,
which a 5-term metalog can only approximate (it overstates this arm's CVaR: about −2.88 vs. the true −2.68).

**So the online metalog's case is not lower regret, it's:**
- constant memory per arm (useful with many arms, or many contexts),
- a full smooth distribution per arm (quantiles, density, samples, any risk measure, not just one CVaR level),
- an estimate that is always a valid distribution.

**Things to try:** other arm distributions (edit `ARMS`), `ALPHA = 0.01`, a smaller budget `T`,
`n_terms=3` or `7` in `MetalogArm.kwargs`, or many more arms, where memory starts to matter.